# 03 · Context re-ranking with XLM-RoBERTa (upper bound for the CPU bigram model)

The app re-ranks spelling suggestions with a bigram model built from 53M tokens of Sindhi text.
This notebook measures how much a masked language model (XLM-R, trained on CC-100 incl. Sindhi)
would add, using the **same held-out test cases**.

**Input:** add the project's `data/kaggle_rerank_input.json` as a Kaggle Dataset
(*Add Input*). It is produced locally by `python scripts/export_rerank_input.py`.

In [ ]:
# Session options (right panel): Accelerator = GPU T4 x2 (or P100), Internet = On.
!pip install -q -U transformers==4.57.1 accelerate sentencepiece sacrebleu ctranslate2==4.6.0 peft datasets
import torch, os
print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

In [ ]:
import glob, json
path = glob.glob("/kaggle/input/**/kaggle_rerank_input.json", recursive=True)[0]
cases = json.load(open(path, encoding="utf-8"))
print(len(cases), cases[0])

In [ ]:
from transformers import AutoTokenizer, AutoModelForMaskedLM
NAME = "FacebookAI/xlm-roberta-base"   # try xlm-roberta-large too (needs ~6 GB)
tok = AutoTokenizer.from_pretrained(NAME)
mlm = AutoModelForMaskedLM.from_pretrained(NAME).half().cuda().eval()

@torch.no_grad()
def pll(sentences):
    """Pseudo-log-likelihood of each sentence (mask one token at a time)."""
    scores = []
    for s in sentences:
        ids = tok(s, return_tensors="pt")["input_ids"][0].cuda()
        positions = list(range(1, len(ids) - 1))
        batch = ids.repeat(len(positions), 1)
        batch[range(len(positions)), positions] = tok.mask_token_id
        logits = mlm(batch).logits.float()
        logp = torch.log_softmax(logits[range(len(positions)), positions], -1)
        scores.append(logp[range(len(positions)), ids[positions]].sum().item())
    return scores

In [ ]:
import numpy as np
hits_base = hits_mlm = hits_mix = 0
for c in cases:
    cands = c["candidates"]                       # top-10 from the app, best first, with its scores
    sents = [f'{c["prev"]} {w["word"]} {c["next"]}' for w in cands]
    lm = np.array(pll(sents))
    app = np.array([-w["score"] for w in cands])  # app score: lower = better
    z = lambda x: (x - x.mean()) / (x.std() + 1e-6)
    hits_base += cands[0]["word"] == c["correct"]
    hits_mlm += cands[int(lm.argmax())]["word"] == c["correct"]
    hits_mix += cands[int((z(app) + 0.5 * z(lm)).argmax())]["word"] == c["correct"]
n = len(cases)
print(f"app (edit distance + freq + bigram): top-1 {hits_base/n:.1%}")
print(f"XLM-R only:                          top-1 {hits_mlm/n:.1%}")
print(f"app + 0.5 * XLM-R:                   top-1 {hits_mix/n:.1%}")